#### Import packages


In [4]:
%load_ext autoreload
%autoreload 2
from __future__ import annotations
import os
import sys
from pathlib import Path
from typing import List, Any, Dict
import numpy as np
import pandas as pd

import shutil
import time
from datetime import datetime

#concurrent/parallel processing
from joblib import Parallel, delayed  # from functools import partial #for specific type of parallel deployment

#load preprocess specific functions
import notebook_setup
info = notebook_setup.setup()
# from run_manifest import create_run_manifest, update_run_database
from config_preprocessing import load_config
from analysis_config_loader import load_analysis_config  # new import, personal config 

preprocess_func_folder = Path(info["repo_root"]) / "preprocess_functions"
sys.path.append(str(preprocess_func_folder)) #add preprocess_functions folder to path to load modules
from detect_cell_enrichment import get_cell_stage_enrichment, get_cell_ensemble_info_per_subject
from matlab_obj_to_python import load_matlab_object, check_corrupted_files

# set up plotting code
import matplotlib as matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sns_plotting_config import * #import dicts containing default plot params


# Apply mplstyle via absolute path from setup info
style_path = Path(info["function_py_storage"]) / "paper_plot.mplstyle"
print(f"Loading style guide at {style_path}")
if style_path.is_file():
    plt.style.use(str(style_path))
else:
    print(f"[warn] Style not found at: {style_path}")

Loading style guide at c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\code\Function .py Storage\paper_plot.mplstyle


#### Define file config variables and analysis parameters ####

In [5]:
# set up locations for input/output 
root_dir = Path(r'c:\\Users\\13car\\Dropbox\\local_github_repos_personal\\dlx56_mPFC_1p_SohalLab\\code')
os.chdir(root_dir)

#import local yaml for env preset variables
config_path = Path(r"analysis_config.yaml")  # adjust as needed
analysis_config = load_analysis_config(config_path)
stage_names = analysis_config.task_phase_names
print(analysis_config)

# Load general file configuration
config = load_config('config.yaml')
data_type = config['data']['data_type_used']
preprocessing = config['preprocessing']
shuffle_cfg = config['shuffles']
print(config)

# Create unique run ID
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
START_TIME = time.time()

# Create output directory with run ID
results_dir = config['data']['results_dir']
run_output_dir = results_dir / 'shuffles' / f"shuffle_run_{RUN_ID}"
run_output_dir.mkdir(parents=True, exist_ok=True)

#set data location
data_dir = config['data']['data_dir']
source_dataset_location= config['data']['source_dataset_location']

#hardcoded/old links for checking
hardcode_results_dir = Path(r"C:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\results")
hardcode_data_dir = Path(r'c:\\Users\\13car\\Dropbox\\local_github_repos_personal\\dlx56_mPFC_1p_SohalLab\\data')
hardcode_source_dataset_location = Path(r"C:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\data\dataset_objects_24-Nov-2024_hour_19") #source dataset location has files created 10-26-2024

#print config paths for verification
print(f"Run ID: {RUN_ID}, Started: {datetime.now()}")
print(f"\nFolders used in run ID: {RUN_ID}")
print(f"Results dir: {results_dir}")
print(f"Data dir: {data_dir}")
print(f"Source dataset location: {source_dataset_location}")

# Copy config to output directory for reference
shutil.copy('config.yaml', run_output_dir / 'config_used.yaml')

AnalysisConfig(trial_section_names=['pre_outcome', 'post_outcome', 'ITI'], phase_division_types=['simple', 'complex'], task_phase_names=['Early_IA_Error', 'Early_IA_Correct', 'Late_IA', 'Early_RS_Error', 'Early_RS_Correct', 'Late_RS'], simple_phase_names=['IA_Error', 'IA_Correct', 'RS_Error', 'RS_Correct'], feature_type_names=['correct_error', 'IA_RS', 'task_phase'], phase_is_solo_vector=True, shuffle_to_use='circle', num_shuffles=1000, early_division_criteria_types=['count', 'first_2'], early_criteria_type='count', early_criteria_value=5, trial_section_divisions=[], corr_vector_is_triangular=True, seconds_before_post_to_keep=5, time_series_bin_size=0.25, corr_time_series_bin_size=0.5, percentile=95, threshold_with_shuffle=True, min_baseline_len=12000, final_thresh=2.5, final_thresh2=12.5, final_thresh3=20, final_thresh4_abs=0.01, drop_low_value_peak_events=False, drop_low_act_cell_in_dataset_obj=False, low_act_thresh_in_obj_init=0.005, cutoff_filter='bimodality', peak_event_cutoff_per

WindowsPath('C:/Users/13car/Dropbox/local_github_repos_personal/dlx56_mPFC_1p_SohalLab/results/shuffles/shuffle_run_20260226_211230/config_used.yaml')

#### Threshold time-series df to remove 0 

In [6]:
from preprocess_data import hyper_param_dict, bin_rotate_timeseries, get_numeric_cols_timeseries, get_unit_mean_timeseries_by_phase, drop_end_bins_of_trials, drop_start_bins_of_trials, add_enriched_in_curr_phase_col, get_subject_stage_info_df
from helper_functions import annotate_csv, get_unit_max_event_rate_of_all_trials, make_folder
from preprocessing_utils import create_subject_trial_tseries_df

<Figure size 960x720 with 0 Axes>

In [7]:
#for concordance with previous code, need to label the stage col as 'task_stage_vec
stage_col = 'task_phase_vec'

def normalize_tseries_df(trial_tseries_df_raw: pd.DataFrame,
                          n_end_timebins_to_drop = config['timeseries']['n_end_timebins_to_drop'], #default = 0,
                          n_start_timebins_to_drop = config['timeseries']['n_start_timebins_to_drop'], #default = 8,
                          name_col: str = 'subject_name',
                          neuron_id_col: str = 'neuron_id',
                          min_max_type: str = 'nonneg',
                          ) -> pd.DataFrame:
    """ Normalize trial time-series DataFrame based on normalization type.
    Inputs:
    trial_tseries_df_raw : DataFrame with raw time-series data (ALREADY BINNED)
    hyper_param_dict : dictionary of hyperparameters
    outcome_post : DataFrame with outcome post data

    Returns: Normalized trial time-series DataFrame
    """
    #get normalization info for curr df
    normed = trial_tseries_df_raw['normalized'].unique()[0]
    if normed == 'none':
        use_min_max_norm = True 
    else:
        use_min_max_norm = False
    print(f" Data Normalization type: {normed} | Applying min-max norm? {use_min_max_norm} ")
    numeric_col = get_numeric_cols_timeseries(trial_tseries_df_raw, " to ") 
    print(numeric_col)
    #is min-max norm run BEFORE or AFTER bin dropping? 
    trial_tseries_df_norm = run_min_max_norm_on_timeseries(use_min_max_norm,
                                                            trial_tseries_df_raw,
                                                              [name_col, neuron_id_col],
                                                                numeric_col, 
                                                                'max_trial_val_all_time') #min_max_norm = True
    
    #To-do add min max normalization flag to either use non-neg timebins or not
    trial_tseries_df_norm['min_max_type'] = min_max_type
    #get mean of each trial- IS this run BEFORE or AFTER norm? 
    use_neg_timebin_in_mean = True #include pre-outcome bins in mean rate calc
    if min_max_type == 'nonneg':
        use_neg_timebin_in_mean = False
        print(f"Ignoring negative timebins in normalization")
    if use_neg_timebin_in_mean:
        trial_tseries_df_norm['mean_rate_in_trial'] =trial_tseries_df_norm[numeric_col].mean(axis = 1)
    else:
        trial_tseries_df_norm['mean_rate_in_trial'] =trial_tseries_df_norm[[c for c in numeric_col if '-' not in c]].mean(axis = 1) #mean rate is post outcome only
    trial_tseries_df_norm['cell_is_active_in_trial'] =trial_tseries_df_norm['mean_rate_in_trial']>0

    #OPTIONAL- drop N bins from start and M bins from end of time-series
    #first, check if you're pre-truncated the data (e.g. already only sliced 3 seconds before outcome)
    neg_timebins = [c for c in numeric_col if "-" in c]
    if len(neg_timebins) <4*3+1: #if you only have 1 second or so of pre-outcome data, you've already sliced 
        n_start_timebins_to_drop = 0
    trial_tseries_df_norm = drop_end_bins_of_trials(trial_tseries_df_norm,numeric_col,  n_end_timebins_to_drop = n_end_timebins_to_drop )
    trial_tseries_df_norm = drop_start_bins_of_trials(trial_tseries_df_norm,numeric_col,  n_start_timebins_to_drop =n_start_timebins_to_drop)
    return trial_tseries_df_norm

def normalize_all_subject_tseries_dfs(raster_base_dfs: Dict[str, pd.DataFrame],
                                      config = config, 
                                      n_end_timebins_to_drop = 0,
                                      n_start_timebins_to_drop = 8,
                                      name_col: str = 'subject_name',
                                      neuron_id_col: str = 'neuron_id',
                                      ) -> pd.DataFrame:
    """ Normalize all subject trial time-series DataFrames.
    Inputs:     raster_base_dfs : Dict of subject_name to raw full recording matrix DataFrame
    optional settings for time-series normalization that equal defaults, so left un input

    Returns: concat. dataframe of subject_name to normalized trial time-series DataFrame
    """

    raster_timeseries =[]
    for name, raster_df in raster_base_dfs.items():
        print(f" Reshaping tseries of {name}: {raster_df.shape}")    
        subj_trial_tseries_df_raw= create_subject_trial_tseries_df(raster_df, dff_ens_matrix, config = config)
        subj_trial_tseries_df_norm= normalize_tseries_df(subj_trial_tseries_df_raw, min_max_type= config['min_max_type'],)
        raster_timeseries.append(subj_trial_tseries_df_norm)
    trial_tseries_df_norm = pd.concat(raster_timeseries)
    return trial_tseries_df_norm

In [8]:
def run_min_max_norm_on_timeseries(run_norm, timeseries_df,
                                    name_unitID_list,
                                    numeric_col,
                                    max_val_col_name = 'max_trial_val_all_time',
                                    min_max_norm_type = 'nonneg'):
    '''
    Docstring for run_min_max_norm_on_timeseries
    
    :param run_norm: Description
    :param timeseries_df: Description
    :param name_unitID_list: list of [subject name col, neuron name col] used in groupby for aggregation
    :param numeric_col: list of cols containing time info 
    :param max_val_col_name: str that sets the name for the maximal event rate reached by that cell, which is defined in *max_e_rate*
    '''

    #get max event rate by unit
    if run_norm:
        groupby_list = name_unitID_list + ['task_phase_vec', 'trial_num']
        max_e_rate = get_unit_max_event_rate_of_all_trials(timeseries_df, groupby_list, numeric_col, name_unitID_list, max_val_col_name)

        if max_val_col_name not in timeseries_df.columns:    #merge timeseries with max event rate by unit record
            nonzero_max_vals = max_e_rate.loc[(max_e_rate[max_val_col_name] > 0),:]
            normed_ts_df = timeseries_df.merge(nonzero_max_vals, how = 'left', on = name_unitID_list) 
        else:
            normed_ts_df = timeseries_df
        # print(f"columns of merge are {normed_ts_df.columns}")
        section_mask = (~normed_ts_df[max_val_col_name].isnull()) & (normed_ts_df[max_val_col_name] > 0) #delete rows if their max val == or nan
        print(f"Dropped {(normed_ts_df[max_val_col_name] == 0).sum()} rows with 0 value max")
        normed_ts_df = normed_ts_df.loc[section_mask,:]
        #normalize by max event rate
        normed_ts_df.loc[:, numeric_col] = normed_ts_df.loc[:, numeric_col].values/normed_ts_df.loc[:, max_val_col_name].values[:,np.newaxis] #find way to avoid NaN values
        #drop rows with no max trial val
        
    else: #else don't norm
        normed_ts_df = timeseries_df 
    return normed_ts_df

#### Import canon ensemble + tseries to compare umap

In [9]:
# import canonical ensemble matrix previously created in MATLAB, used for paper #find old file- 
canonical_file = data_dir / Path(r"Dlx56_Normalized Trial Calcium Timeseries_20_Jun_2025.parquet")
canonical_data = pd.read_parquet(canonical_file) #normalized spike rates for the means 
print(canonical_data.info())

canonical_data.tail()

<class 'pandas.core.frame.DataFrame'>
Index: 93253 entries, 0 to 116376
Data columns (total 93 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   task_phase_vec     93253 non-null  object 
 1   IA_RS_vec          93253 non-null  object 
 2   corr_err_vec       93253 non-null  object 
 3   trial_num          93253 non-null  int64  
 4   neuron_ID          93253 non-null  int64  
 5   name               93253 non-null  object 
 6   day                93253 non-null  int32  
 7   geno               93253 non-null  object 
 8   geno_day           93253 non-null  object 
 9   -3.0s to -2.75s    93253 non-null  float64
 10  -2.75s to -2.5s    93253 non-null  float64
 11  -2.5s to -2.25s    93253 non-null  float64
 12  -2.25s to -2.0s    93253 non-null  float64
 13  -2.0s to -1.75s    93253 non-null  float64
 14  -1.75s to -1.5s    93253 non-null  float64
 15  -1.5s to -1.25s    93253 non-null  float64
 16  -1.25s to -1.0s    93253 n

,task_phase_vec,IA_RS_vec,corr_err_vec,trial_num,neuron_ID,name,day,geno,geno_day,-3.0s to -2.75s,...,Late_IA,Early_RS_Error,Early_RS_Correct,Late_RS,enriched_in_phase,any_enrichment,unique_ID,max_trial_val,mean_rate,active_in_trial
116370,Late_RS,RS,Correct,38,209,9_3_HET_RS3,3,HET,Het postCLNZ,0.0,...,0.0,0.0,0.0,0.0,False,1.0,9_3_HET_RS3-209,1.0,0.02,True
116373,Late_RS,RS,Correct,38,212,9_3_HET_RS3,3,HET,Het postCLNZ,0.0,...,0.0,0.0,0.0,0.0,False,2.0,9_3_HET_RS3-212,1.0,0.03,True
116374,Late_RS,RS,Correct,38,213,9_3_HET_RS3,3,HET,Het postCLNZ,0.0,...,0.0,0.0,0.0,0.0,False,0.0,9_3_HET_RS3-213,1.0,0.00,False
116375,Late_RS,RS,Correct,38,214,9_3_HET_RS3,3,HET,Het postCLNZ,0.0,...,0.0,0.0,0.0,0.0,False,0.0,9_3_HET_RS3-214,1.0,0.20,True
116376,Late_RS,RS,Correct,38,215,9_3_HET_RS3,3,HET,Het postCLNZ,0.0,...,0.0,0.0,0.0,0.0,False,0.0,9_3_HET_RS3-215,0.8,0.00,False


In [10]:
#get canon mean unit timeseries
numeric_col_canon = get_numeric_cols_timeseries(canonical_data, " to ")  #update post drop\
canon_mean_unit_tseries  =  get_unit_mean_timeseries_by_phase(canonical_data,['trial_num'], ['name', 'neuron_ID','geno_day', stage_col], numeric_col_canon) 
print(canon_mean_unit_tseries.groupby(by = 'geno_day')['unique_ID'].nunique())
print(canon_mean_unit_tseries.info())
canon_mean_unit_tseries.tail()


geno_day
Het CLNZ         866
Het VEH          874
Het postCLNZ     937
WT CLNZ         1019
WT VEH          1080
Name: unique_ID, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27768 entries, 0 to 27767
Data columns (total 94 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   name               27768 non-null  object 
 1   neuron_ID          27768 non-null  int64  
 2   geno_day           27768 non-null  object 
 3   task_phase_vec     27768 non-null  object 
 4   -3.0s to -2.75s    27768 non-null  float64
 5   -2.75s to -2.5s    27768 non-null  float64
 6   -2.5s to -2.25s    27768 non-null  float64
 7   -2.25s to -2.0s    27768 non-null  float64
 8   -2.0s to -1.75s    27768 non-null  float64
 9   -1.75s to -1.5s    27768 non-null  float64
 10  -1.5s to -1.25s    27768 non-null  float64
 11  -1.25s to -1.0s    27768 non-null  float64
 12  -1.0s to -0.75s    27768 non-null  float64
 13  -0.75s to -0.5s    277

,name,neuron_ID,geno_day,task_phase_vec,-3.0s to -2.75s,-2.75s to -2.5s,-2.5s to -2.25s,-2.25s to -2.0s,-2.0s to -1.75s,-1.75s to -1.5s,...,Early_RS_Correct,Late_RS,enriched_in_phase,any_enrichment,unique_ID,max_trial_val,mean_rate,active_in_trial,max_val,max_val_tbin
27763,9_3_HET_RS3,215,Het postCLNZ,Early_IA_Error,0.00,0.00,0.0,0.0,0.0,0.0000,...,0.0,0.0,False,0.0,9_3_HET_RS3-215,0.8,0.000000,False,0.0000,-3.0s to -2.75s
27764,9_3_HET_RS3,215,Het postCLNZ,Early_RS_Correct,0.00,0.00,0.0,0.0,0.0,0.0000,...,0.0,0.0,False,0.0,9_3_HET_RS3-215,0.8,0.000000,False,0.0000,-3.0s to -2.75s
27765,9_3_HET_RS3,215,Het postCLNZ,Early_RS_Error,0.00,0.00,0.0,0.0,0.0,0.0000,...,0.0,0.0,False,0.0,9_3_HET_RS3-215,0.8,0.000000,False,0.0000,-3.0s to -2.75s
27766,9_3_HET_RS3,215,Het postCLNZ,Late_IA,0.05,0.00,0.2,0.0,0.2,0.0500,...,0.0,0.0,False,0.0,9_3_HET_RS3-215,0.8,0.000000,False,0.2000,-2.5s to -2.25s
27767,9_3_HET_RS3,215,Het postCLNZ,Late_RS,0.00,0.25,0.0,0.0,0.0,0.0625,...,0.0,0.0,False,0.0,9_3_HET_RS3-215,0.8,0.021875,False,0.3125,1.25s to 1.5s


#### |NEW- import 10K matlab shuffle run

In [11]:
# Load the raw files first
# matlab_TACO_filename = data_dir / Path(r"29-Jan-2026_neurons_ sig enrichment vectors by task stage_10000__shuffles_hour__15\WT_CLNZ_complexTACO- neurons_activity by task phase_30-Jan-2026.xls")
matlab_TACO_filename = data_dir / Path(r'DfF MATLAB 1K shuffle time-series & ensemble\WT_CLNZ_complexTACO- neurons_dff_zscore_activity by task phase_25-Feb-2026.xls')
# matlab_timeseries_filename = data_dir / Path(r"30-Jan-2026_neurons_trial activity timeseries data_hour_18\post_outcome_WT_CLNZ_neurons_trial activity timeseries data_30-Jan-2026_timeseries.csv")
matlab_timeseries_filename = data_dir / Path(r'DfF MATLAB 1K shuffle time-series & ensemble\post_outcome_WT_CLNZ_neurons__dff_zscore_trial activity timeseries data_25-Feb-2026_timeseries.csv')

In [12]:
input_data_normed = {True: 'zscored', False:'none'}
input_data_zscored = True
print(input_data_normed[input_data_zscored])

zscored


In [13]:
# Check source files- TACO
cell_period_active = pd.read_excel(matlab_TACO_filename, na_values=[-99, '-99']).replace(-99, np.nan)
generic_col_to_drop = [c for c in cell_period_active.columns if any(p in c for p in ('threshold', 'peak', 'ITI', 'cutoff', 'pre_outcome'))]
print(f" dropping cols: {generic_col_to_drop}")
cell_period_active.drop(columns = generic_col_to_drop + ['Var1_4', 'Var1_5', 'Var1_6'], inplace = True)
cell_period_active = cell_period_active.rename(columns = {'name': 'subject_name', 'Var1_1': "Run_end_time", 'Var1_2': "num_shuffles", 'Var1_3':"percentile_cutoff",
                                                          **{'post_outcome_'+s:s for s in stage_names}})
cell_period_active['session'] = cell_period_active['subject_name'].str.split("_").str.get(3)
## add neuron Id unique col 
cell_period_active['raw_ID'] = cell_period_active['neuron_ID'].astype(str)
cell_col = 'neuron_ID'
cell_period_active[cell_col] = cell_period_active['subject_name'] + '-' + cell_period_active['raw_ID']

print(cell_period_active.info())
cell_period_active.head()


 dropping cols: ['pre_outcome_Early_IA_Error', 'pre_outcome_Early_IA_Correct', 'pre_outcome_Late_IA', 'pre_outcome_Early_RS_Error', 'pre_outcome_Early_RS_Correct', 'pre_outcome_Late_RS', 'ITI_Early_IA_Error', 'ITI_Early_IA_Correct', 'ITI_Late_IA', 'ITI_Early_RS_Error', 'ITI_Early_RS_Correct', 'ITI_Late_RS', 'pvalue_pre_outcome_Early_IA_Error', 'pvalue_pre_outcome_Early_IA_Correct', 'pvalue_pre_outcome_Late_IA', 'pvalue_pre_outcome_Early_RS_Error', 'pvalue_pre_outcome_Early_RS_Correct', 'pvalue_pre_outcome_Late_RS', 'pvalue_ITI_Early_IA_Error', 'pvalue_ITI_Early_IA_Correct', 'pvalue_ITI_Late_IA', 'pvalue_ITI_Early_RS_Error', 'pvalue_ITI_Early_RS_Correct', 'pvalue_ITI_Late_RS', 'threshold_95_pre_outcome_Early_IA_Error', 'threshold_95_pre_outcome_Early_IA_Correct', 'threshold_95_pre_outcome_Late_IA', 'threshold_95_pre_outcome_Early_RS_Error', 'threshold_95_pre_outcome_Early_RS_Correct', 'threshold_95_pre_outcome_Late_RS', 'threshold_95_post_outcome_Early_IA_Error', 'threshold_95_post_outc

,Early_IA_Error,Early_IA_Correct,Late_IA,Early_RS_Error,Early_RS_Correct,Late_RS,pvalue_post_outcome_Early_IA_Error,pvalue_post_outcome_Early_IA_Correct,pvalue_post_outcome_Late_IA,pvalue_post_outcome_Early_RS_Error,...,baseline_sig_cells,subject_name,geno_day,geno,neuron_ID,Run_end_time,num_shuffles,percentile_cutoff,session,raw_ID
0,1.0,0,0,0,0,0,0.003,0.440,0.488,0.677,...,False,10_3_HET_RS1,HET RS1,HET,10_3_HET_RS1-1,25-Feb-2026 16:02:15,1000,95,RS1,1
1,0.0,0,0,0,0,0,0.508,0.710,0.616,0.576,...,False,10_3_HET_RS1,HET RS1,HET,10_3_HET_RS1-2,25-Feb-2026 16:02:15,1000,95,RS1,2
2,1.0,0,0,0,0,0,0.036,0.584,0.199,0.097,...,True,10_3_HET_RS1,HET RS1,HET,10_3_HET_RS1-3,25-Feb-2026 16:02:15,1000,95,RS1,3
3,0.0,0,0,0,0,0,0.418,0.461,0.138,0.232,...,True,10_3_HET_RS1,HET RS1,HET,10_3_HET_RS1-4,25-Feb-2026 16:02:15,1000,95,RS1,4
4,0.0,0,0,0,1,0,1.000,0.059,0.431,0.069,...,False,10_3_HET_RS1,HET RS1,HET,10_3_HET_RS1-5,25-Feb-2026 16:02:15,1000,95,RS1,5


In [14]:
print(f"cell_period_active[stage_names].value_counts() = {cell_period_active[stage_names].value_counts()}")

cell_period_active[stage_names].value_counts() = Early_IA_Error  Early_IA_Correct  Late_IA  Early_RS_Error  Early_RS_Correct  Late_RS
0.0             0                 0        0               0                 0          3309
1.0             0                 0        0               0                 0           293
0.0             1                 0        0               0                 0           255
                0                 1        0               0                 0           153
                                  0        1               0                 0           131
                                           0               1                 0           113
                                                           0                 1            75
1.0             1                 0        0               0                 0            71
0.0             1                 1        0               0                 0            54
                             

In [15]:
## import outcome post tseries
outcome_post = pd.read_csv(matlab_timeseries_filename, header=0, low_memory=False).rename(columns = {'name': 'subject_name'}).dropna(subset = [stage_col]).replace(-99, np.nan)
outcome_post['session'] = outcome_post['subject_name'].str.split("_").str.get(3)
outcome_post['raw_ID']= outcome_post[cell_col].astype(str)
outcome_post[cell_col] = outcome_post['subject_name'] + '-' + outcome_post['raw_ID']

print(outcome_post.info())
outcome_post.head()


<class 'pandas.core.frame.DataFrame'>
Index: 116377 entries, 0 to 191420
Columns: 416 entries, task_phase_vec to raw_ID
dtypes: float64(401), int64(6), object(9)
memory usage: 370.2+ MB
None


,task_phase_vec,IA_RS_vec,corr_err_vec,f_1,f_2,f_3,f_4,f_5,f_6,f_7,...,-f_8,-f_7,-f_6,-f_5,-f_4,-f_3,-f_2,-f_1,session,raw_ID
0,Early_IA_Correct,IA,Correct,-0.214444,-0.214444,-0.214444,-0.214444,-0.214444,-0.214444,-0.214444,...,-0.214444,-0.214444,-0.214444,-0.214444,-0.214444,-0.214444,-0.214444,-0.214444,RS1,1
1,Early_IA_Correct,IA,Correct,-0.367535,-0.367535,-0.367535,-0.367535,-0.367535,-0.367535,-0.367535,...,-0.367535,-0.367535,-0.367535,-0.367535,-0.367535,-0.367535,-0.367535,-0.367535,RS1,2
2,Early_IA_Correct,IA,Correct,-0.258050,-0.258050,-0.258050,-0.258050,-0.258050,-0.258050,-0.258050,...,-0.258050,-0.258050,-0.258050,-0.258050,-0.195513,-0.258050,-0.258050,-0.258050,RS1,3
3,Early_IA_Correct,IA,Correct,-0.385787,-0.385787,-0.385787,-0.385787,-0.385787,-0.385787,-0.385787,...,-0.385787,-0.385787,-0.385787,-0.385787,-0.385787,-0.385787,-0.385787,-0.385787,RS1,4
4,Early_IA_Correct,IA,Correct,1.179336,1.372140,1.666922,0.862826,1.309123,1.054217,0.849840,...,0.320787,-0.137262,-0.260030,-0.026771,-0.260030,-0.260030,-0.049654,-0.181813,RS1,5


In [16]:
window_to_bin = 5
n_sec_to_rotate = 0
## drop + annotate
outcome_col_to_drop = ['threshold_with_shuffle', 'n_trial_frames', 'drop_low_value_peak_events','peak_dff_threshold_percentile', 'cutoff_filter', 'peak_event_cutoff_percentile']
outcome_post.drop(columns = [c for c in outcome_col_to_drop if c in outcome_post.columns], inplace = True)
annotate_csv(outcome_post, 'subject_name')
outcome_post[cell_col] = outcome_post[cell_col].astype(str)
# Run this after create_subject_trial_tseries_df but before bin_rotate_timeseries # Get frame columns
fr_col = outcome_post.columns[outcome_post.columns.str.contains('f_')]
neg_frames = fr_col[fr_col.str.contains('-')].tolist()
pos_frames = fr_col[~fr_col.str.contains('-')].tolist()
combined = neg_frames + pos_frames

## begin preprocess- bin and rotate timeseries 
outcome_post = bin_rotate_timeseries(outcome_post, window_size = window_to_bin, rotate_by = n_sec_to_rotate)
outcome_post.tail()


binning data- post. # post-outcome frames: 300 #pre-outcome frames: 101
rem_frames: 1
Merge penultimate/last bin with only 1 frame in it, then dropping last bin
 Moving numeric coluns by 0 bins


,task_phase_vec,IA_RS_vec,corr_err_vec,trial_num,neuron_ID,section,subject_name,session,raw_ID,day,...,12.5s to 12.75s,12.75s to 13.0s,13.0s to 13.25s,13.25s to 13.5s,13.5s to 13.75s,13.75s to 14.0s,14.0s to 14.25s,14.25s to 14.5s,14.5s to 14.75s,14.75s to 15.0s
191416,Late_RS,RS,Correct,38,9_3_HET_RS3-211,post_outcome,9_3_HET_RS3,RS3,211,3,...,0.333632,-0.103866,-0.418694,-0.377777,0.012962,-0.289741,0.146990,0.128020,1.071950,-0.144368
191417,Late_RS,RS,Correct,38,9_3_HET_RS3-212,post_outcome,9_3_HET_RS3,RS3,212,3,...,0.087290,-0.044948,-0.068705,-0.055996,-0.104207,0.003766,-0.136562,-0.098727,-0.039455,-0.107218
191418,Late_RS,RS,Correct,38,9_3_HET_RS3-213,post_outcome,9_3_HET_RS3,RS3,213,3,...,-0.079766,-0.358463,0.316222,0.002399,0.947768,0.907298,-0.207141,1.473512,0.228865,1.006132
191419,Late_RS,RS,Correct,38,9_3_HET_RS3-214,post_outcome,9_3_HET_RS3,RS3,214,3,...,-0.322511,-0.322511,-0.322511,-0.322511,-0.322511,-0.322511,-0.322511,-0.161303,-0.322511,-0.285947
191420,Late_RS,RS,Correct,38,9_3_HET_RS3-215,post_outcome,9_3_HET_RS3,RS3,215,3,...,-0.345884,0.448927,-0.266699,-0.415356,0.025681,-0.078277,0.024783,-0.320970,-0.415356,0.315419


In [17]:
TACO_cols_to_keep = ['Early_IA_Error', 'Early_IA_Correct', 'Late_IA', 'Early_RS_Error','Early_RS_Correct', 'Late_RS',
         'pvalue_post_outcome_Early_IA_Error', 'pvalue_post_outcome_Early_IA_Correct', 'pvalue_post_outcome_Late_IA',
         'pvalue_post_outcome_Early_RS_Error','pvalue_post_outcome_Early_RS_Correct', 'pvalue_post_outcome_Late_RS',
         'neuron_ID','Run_end_time', 'num_shuffles']

outcome_post_with_enrichment = outcome_post.merge(cell_period_active.loc[:,TACO_cols_to_keep], on='neuron_ID', how='left', suffixes=('', '_ens')) #join  timeseries with ensemble info
# Check that row count is preserved (should be same as outcome_post)
print(f"outcome_post rows: {len(outcome_post)}. merged rows: {len(outcome_post_with_enrichment)}")
#drop time-series not belonging to any task stage , aka the 0 task stage 
outcome_post_with_enrichment['any_enrichment']= outcome_post_with_enrichment[stage_names].sum(axis = 1)
outcome_post_with_enrichment = outcome_post_with_enrichment[outcome_post_with_enrichment[stage_col].isin(stage_names)]

## make raw timeseries
trial_tseries_df_raw = add_enriched_in_curr_phase_col(outcome_post_with_enrichment, stage_col)
trial_tseries_df_raw['normalized'] = input_data_normed[input_data_zscored]
trial_tseries_df_raw.head()

outcome_post rows: 116377. merged rows: 116377


,task_phase_vec,IA_RS_vec,corr_err_vec,trial_num,neuron_ID,section,subject_name,session,raw_ID,day,...,pvalue_post_outcome_Early_IA_Correct,pvalue_post_outcome_Late_IA,pvalue_post_outcome_Early_RS_Error,pvalue_post_outcome_Early_RS_Correct,pvalue_post_outcome_Late_RS,Run_end_time,num_shuffles,any_enrichment,enriched_in_phase,normalized
0,Early_IA_Correct,IA,Correct,1,10_3_HET_RS1-1,post_outcome,10_3_HET_RS1,RS1,1,1,...,0.440,0.488,0.677,0.201,0.758,25-Feb-2026 16:02:15,1000,1.0,False,zscored
1,Early_IA_Correct,IA,Correct,1,10_3_HET_RS1-2,post_outcome,10_3_HET_RS1,RS1,2,1,...,0.710,0.616,0.576,0.770,0.290,25-Feb-2026 16:02:15,1000,0.0,False,zscored
2,Early_IA_Correct,IA,Correct,1,10_3_HET_RS1-3,post_outcome,10_3_HET_RS1,RS1,3,1,...,0.584,0.199,0.097,0.387,0.381,25-Feb-2026 16:02:15,1000,1.0,False,zscored
3,Early_IA_Correct,IA,Correct,1,10_3_HET_RS1-4,post_outcome,10_3_HET_RS1,RS1,4,1,...,0.461,0.138,0.232,0.955,0.934,25-Feb-2026 16:02:15,1000,0.0,False,zscored
4,Early_IA_Correct,IA,Correct,1,10_3_HET_RS1-5,post_outcome,10_3_HET_RS1,RS1,5,1,...,0.059,0.431,0.069,0.047,0.732,25-Feb-2026 16:02:15,1000,1.0,False,zscored


In [18]:
## normalize timeseries
trial_tseries_df_norm = normalize_tseries_df(trial_tseries_df_raw,neuron_id_col= "neuron_ID", min_max_type= 'all_frames',)
trial_tseries_df_norm

 Data Normalization type: zscored | Applying min-max norm? False 
Index(['-5.0s to -4.75s', '-4.75s to -4.5s', '-4.5s to -4.25s',
       '-4.25s to -4.0s', '-4.0s to -3.75s', '-3.75s to -3.5s',
       '-3.5s to -3.25s', '-3.25s to -3.0s', '-3.0s to -2.75s',
       '-2.75s to -2.5s', '-2.5s to -2.25s', '-2.25s to -2.0s',
       '-2.0s to -1.75s', '-1.75s to -1.5s', '-1.5s to -1.25s',
       '-1.25s to -1.0s', '-1.0s to -0.75s', '-0.75s to -0.5s',
       '-0.5s to -0.25s', '-0.25s to -0.0s', '0.0s to 0.25s', '0.25s to 0.5s',
       '0.5s to 0.75s', '0.75s to 1.0s', '1.0s to 1.25s', '1.25s to 1.5s',
       '1.5s to 1.75s', '1.75s to 2.0s', '2.0s to 2.25s', '2.25s to 2.5s',
       '2.5s to 2.75s', '2.75s to 3.0s', '3.0s to 3.25s', '3.25s to 3.5s',
       '3.5s to 3.75s', '3.75s to 4.0s', '4.0s to 4.25s', '4.25s to 4.5s',
       '4.5s to 4.75s', '4.75s to 5.0s', '5.0s to 5.25s', '5.25s to 5.5s',
       '5.5s to 5.75s', '5.75s to 6.0s', '6.0s to 6.25s', '6.25s to 6.5s',
       '6.5s to 6.75s

,task_phase_vec,IA_RS_vec,corr_err_vec,trial_num,neuron_ID,section,subject_name,session,raw_ID,day,...,pvalue_post_outcome_Early_RS_Correct,pvalue_post_outcome_Late_RS,Run_end_time,num_shuffles,any_enrichment,enriched_in_phase,normalized,min_max_type,mean_rate_in_trial,cell_is_active_in_trial
0,Early_IA_Correct,IA,Correct,1,10_3_HET_RS1-1,post_outcome,10_3_HET_RS1,RS1,1,1,...,0.201,0.758,25-Feb-2026 16:02:15,1000,1.0,False,zscored,all_frames,-0.214444,False
1,Early_IA_Correct,IA,Correct,1,10_3_HET_RS1-2,post_outcome,10_3_HET_RS1,RS1,2,1,...,0.770,0.290,25-Feb-2026 16:02:15,1000,0.0,False,zscored,all_frames,-0.367535,False
2,Early_IA_Correct,IA,Correct,1,10_3_HET_RS1-3,post_outcome,10_3_HET_RS1,RS1,3,1,...,0.387,0.381,25-Feb-2026 16:02:15,1000,1.0,False,zscored,all_frames,-0.215792,False
3,Early_IA_Correct,IA,Correct,1,10_3_HET_RS1-4,post_outcome,10_3_HET_RS1,RS1,4,1,...,0.955,0.934,25-Feb-2026 16:02:15,1000,0.0,False,zscored,all_frames,-0.372400,False
4,Early_IA_Correct,IA,Correct,1,10_3_HET_RS1-5,post_outcome,10_3_HET_RS1,RS1,5,1,...,0.047,0.732,25-Feb-2026 16:02:15,1000,1.0,False,zscored,all_frames,0.277849,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116372,Late_RS,RS,Correct,38,9_3_HET_RS3-211,post_outcome,9_3_HET_RS3,RS3,211,3,...,0.042,0.376,25-Feb-2026 16:02:15,1000,1.0,False,zscored,all_frames,0.224504,True
116373,Late_RS,RS,Correct,38,9_3_HET_RS3-212,post_outcome,9_3_HET_RS3,RS3,212,3,...,0.193,0.319,25-Feb-2026 16:02:15,1000,2.0,False,zscored,all_frames,-0.079418,False
116374,Late_RS,RS,Correct,38,9_3_HET_RS3-213,post_outcome,9_3_HET_RS3,RS3,213,3,...,0.531,0.328,25-Feb-2026 16:02:15,1000,0.0,False,zscored,all_frames,0.309805,True
116375,Late_RS,RS,Correct,38,9_3_HET_RS3-214,post_outcome,9_3_HET_RS3,RS3,214,3,...,0.795,0.760,25-Feb-2026 16:02:15,1000,0.0,False,zscored,all_frames,0.042397,True


In [19]:
print(trial_tseries_df_norm.groupby('geno_day')['neuron_ID'].nunique())
trial_tseries_df_norm.groupby('geno_day')['neuron_ID'].nunique().sum()

geno_day
Het CLNZ        1094
Het VEH         1094
Het postCLNZ    1177
WT CLNZ         1246
WT VEH          1350
Name: neuron_ID, dtype: int64


5961

In [20]:
n_shuff = trial_tseries_df_norm.num_shuffles.max()
new_timeseries_save_folder = data_dir / Path(f"matlab_import_{n_shuff}_shuff_minmax_normed_trial_tseries_{datetime.now().strftime('%d-%b-%Y')}")
make_folder(new_timeseries_save_folder)

#save filename with data type and date info
save_name = new_timeseries_save_folder / Path(f"dff_matlab-made_trial_timeseries_normalized_{datetime.now().strftime("%Y-%m-%d %H")}.parquet")
print(f"saving file as {save_name}")
trial_tseries_df_norm.to_parquet(save_name)

The folder 'c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\data\matlab_import_1000_shuff_minmax_normed_trial_tseries_26-Feb-2026' already exists.
saving file as c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\data\matlab_import_1000_shuff_minmax_normed_trial_tseries_26-Feb-2026\dff_matlab-made_trial_timeseries_normalized_2026-02-26 21.parquet


#### Save time-series and record of config used

In [21]:
## get 2025 matlab time-series

new_matlab_post_file = data_dir/ Path(r"post_outcome_WT_CLNZ_neurons_trial activity timeseries data_18-Nov-2025_timeseries.csv")
new_matlab_post_timeseries = pd.read_csv(new_matlab_post_file)
new_matlab_post_metadata_cols = [c for c in new_matlab_post_timeseries.columns if "f_" not in c]

new_matlab_post_timeseries['unique_ID'] = new_matlab_post_timeseries['name'].astype(str) + "-" +new_matlab_post_timeseries['neuron_ID'].astype(str)
new_matlab_post_timeseries

C:\Users\13car\AppData\Local\Temp\ipykernel_77748\976479117.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  new_matlab_post_timeseries = pd.read_csv(new_matlab_post_file)


,task_phase_vec,IA_RS_vec,corr_err_vec,f_1,f_2,f_3,f_4,f_5,f_6,f_7,...,-f_9,-f_8,-f_7,-f_6,-f_5,-f_4,-f_3,-f_2,-f_1,unique_ID
0,Early_IA_Correct,IA,Correct,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,10_3_HET_RS1-1
1,Early_IA_Correct,IA,Correct,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,10_3_HET_RS1-2
2,Early_IA_Correct,IA,Correct,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,10_3_HET_RS1-3
3,Early_IA_Correct,IA,Correct,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,10_3_HET_RS1-4
4,Early_IA_Correct,IA,Correct,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,10_3_HET_RS1-5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191416,Late_RS,RS,Correct,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,9_3_HET_RS3-211
191417,Late_RS,RS,Correct,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,9_3_HET_RS3-212
191418,Late_RS,RS,Correct,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,9_3_HET_RS3-213
191419,Late_RS,RS,Correct,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,9_3_HET_RS3-214


In [22]:
#set shuffle storage data folder
run_output_folder = results_dir / Path(f"{data_type_used}_{normalize}_norm_ensemble_detection_{datetime.now().strftime('%d-%b-%Y')}_{n_shuf_per_subject} shuffles")
print(f" saving to {run_output_folder}")
run_output_folder.mkdir(parents=True, exist_ok=True)

new_timeseries_save_folder = run_output_folder / Path(f"{data_type_used}_{min_max_type}_minmaxnormed_trial_tseries_{datetime.now().strftime('%d-%b-%Y')}")
make_folder(new_timeseries_save_folder)

#save filename with data type and date info
save_name = data_dir / Path(f"python-made_trial_timeseries_{min_max_type}_minmax_{data_type_used}_{datetime.now().strftime("%Y-%m-%d %H")}.parquet")
print(f"saving file as {save_name}")
trial_tseries_df_norm.to_parquet(save_name)


NameError: name 'data_type_used' is not defined